# A/B Test Analysis — E-Commerce Checkout Redesign
### Data Analyst Portfolio Project — Istanbul Job Market

---

## What is this project about?

Every tech company runs A/B tests constantly. Trendyol, Hepsiburada, Getir, and Vodafone Turkey all make product decisions based on them. An A/B test answers a very specific question:

> "If we change X, does it actually improve Y — or did we just get lucky?"

For example: we redesigned the checkout page. Did conversion rate go up because the new design is better, or did it happen to be a good week for sales anyway?

**This is the most commonly tested data analyst skill in interviews.** Interviewers will give you a dataset and ask you to tell them whether the test worked. This notebook shows you how to do that properly — and how to explain the result to a non-technical product manager.

## The scenario

We are analysts at a Turkish e-commerce platform (think: Trendyol). The product team redesigned the checkout flow to reduce the number of steps from 4 to 2. They ran the new design on 50% of users for 2 weeks.

**The question:** Did the new checkout increase the conversion rate (percentage of users who complete a purchase)?

## What this notebook covers

| Step | Topic | Why it matters |
|---|---|---|
| 1 | Experiment design | How to set up a test correctly before running it |
| 2 | Sample size & power | How many users do you need? |
| 3 | Data sanity checks | Is the data trustworthy? |
| 4 | Statistical test | Is the result real or luck? |
| 5 | Practical significance | Is the result big enough to matter? |
| 6 | Segmentation | Does the result hold across all user groups? |
| 7 | Business recommendation | What do we actually do? |

---
**References:**
- Kohavi et al. (2020) *Trustworthy Online Controlled Experiments* — the definitive A/B testing book
- Deng et al. (2013) *Improving the Sensitivity of Online Controlled Experiments* — Microsoft Research

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import chi2_contingency, norm, ttest_ind, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d', 'grid.color': '#21262d',
    'axes.labelcolor': '#c9d1d9', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'text.color': '#c9d1d9', 'font.size': 11
})

SEED = 42
np.random.seed(SEED)
print('Ready.')

## Step 1 — Experiment Design

### The most important step happens BEFORE you collect any data

A badly designed experiment gives results you cannot trust, no matter how good your statistics are. Before running any A/B test, you must define:

**1. The metric (what you measure)**
Our primary metric is **conversion rate** — the percentage of users who complete a purchase after starting checkout. This is a binary outcome: each user either converts (1) or does not (0).

Secondary metrics we also track:
- Average order value (AOV) — does the new checkout change how much people spend?
- Time to complete checkout — is the new flow actually faster?
- Cart abandonment rate — are users dropping off at different stages?

**2. The hypothesis**
- **Null hypothesis (H₀):** The new checkout has no effect on conversion rate
- **Alternative hypothesis (H₁):** The new checkout changes conversion rate

We use a **two-tailed test** because we want to detect both improvements AND regressions. If the new checkout is worse, we need to know that too.

**3. Significance level (α) and power (1-β)**
- α = 0.05 means: we accept a 5% chance of declaring a winner when there is none (false positive / Type I error)
- Power = 0.80 means: we want an 80% chance of detecting a real effect if one exists (true positive rate)

These are industry standard values. Raising power to 0.90 would require ~35% more users.

**4. Minimum Detectable Effect (MDE)**
The smallest improvement that would be worth launching. If conversion rate is currently 3.5%, a 0.01% improvement is meaningless — engineering cost is not worth it. We set MDE = 0.5 percentage points (a relative improvement of ~14%).

Setting the MDE before the experiment prevents **p-hacking**: if you only launch when you see an improvement, you are biasing your results.

In [ ]:
# ── Sample Size Calculation ───────────────────────────────────
# This is done BEFORE the experiment starts

p_baseline = 0.035   # current conversion rate: 3.5%
mde        = 0.005   # minimum detectable effect: +0.5 percentage points
p_new      = p_baseline + mde   # expected conversion under treatment
alpha      = 0.05    # significance level
power      = 0.80    # desired statistical power

def calculate_sample_size(p1, p2, alpha=0.05, power=0.80):
    """
    Calculate required sample size per group for a two-proportion z-test.

    Formula derivation:
    We need enough users so that if the true effect is MDE,
    we have `power` probability of detecting it at significance `alpha`.

    z_alpha/2 = critical value for significance (1.96 for alpha=0.05)
    z_beta    = critical value for power (0.84 for power=0.80)

    n = (z_alpha/2 + z_beta)^2 * (p1*(1-p1) + p2*(1-p2)) / (p1-p2)^2
    """
    z_alpha = norm.ppf(1 - alpha / 2)   # two-tailed: 1.96
    z_beta  = norm.ppf(power)           # 0.842
    pooled_var = p1*(1-p1) + p2*(1-p2)
    n = (z_alpha + z_beta)**2 * pooled_var / (p2 - p1)**2
    return int(np.ceil(n))

n_per_group = calculate_sample_size(p_baseline, p_new, alpha, power)
n_total     = n_per_group * 2

print('='*55)
print('EXPERIMENT DESIGN SUMMARY')
print('='*55)
print(f'Control conversion rate:    {p_baseline:.1%}')
print(f'Minimum detectable effect:  +{mde:.1%} ({mde/p_baseline*100:.0f}% relative improvement)')
print(f'Significance level (α):     {alpha}')
print(f'Statistical power (1-β):    {power}')
print(f'\nRequired sample size:')
print(f'  Per group:  {n_per_group:,} users')
print(f'  Total:      {n_total:,} users')
print()

# How long will this take?
daily_users_in_checkout = 15000  # realistic for a mid-size Turkish e-commerce
days_needed = n_total / (daily_users_in_checkout * 0.5)  # 50% split
print(f'At {daily_users_in_checkout:,} daily checkout users (50/50 split):')
print(f'  Days needed: {days_needed:.0f} days (~{days_needed/7:.1f} weeks)')
print()
print('Important: run for full weeks to avoid day-of-week bias.')
print('If you stop early because results look good, you are p-hacking.')

# Show sample size sensitivity
fig, ax = plt.subplots(figsize=(14, 5))
mdes = np.arange(0.001, 0.015, 0.001)
sizes = [calculate_sample_size(p_baseline, p_baseline+m, alpha, power) for m in mdes]
ax.plot(mdes*100, sizes, color='#f5c842', lw=2.5, marker='o', ms=6)
ax.axvline(mde*100, color='#f06090', lw=2, ls='--', label=f'Our MDE = {mde*100:.1f}pp → {n_per_group:,} per group')
ax.set_xlabel('Minimum Detectable Effect (percentage points)')
ax.set_ylabel('Required sample size per group')
ax.set_title('Sample Size vs MDE — the smaller the effect you want to detect, the more users you need',
              fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('ab_design.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Step 2 — Simulating the Experiment Data

We simulate 2 weeks of experiment data. Each row represents one user who entered the checkout flow. We record:
- Which group they were in (control = old checkout, treatment = new checkout)
- Whether they converted
- Their order value (if they converted)
- How long checkout took
- Their device type, new vs returning status, day of week

We set the true treatment effect to +0.6 percentage points (slightly above our MDE of 0.5pp), so we expect to detect it — but it will not be overwhelmingly obvious from the raw numbers.

In [ ]:
np.random.seed(SEED)

# True effect we inject (hidden from the analysis perspective)
TRUE_EFFECT = 0.006   # +0.6 percentage points
p_control   = p_baseline
p_treatment = p_baseline + TRUE_EFFECT

N_CONTROL   = n_per_group
N_TREATMENT = n_per_group
N_TOTAL_SIM = N_CONTROL + N_TREATMENT

# Generate user-level data
days      = np.random.choice(range(14), N_TOTAL_SIM)  # 2-week experiment
group     = np.array(['control']*N_CONTROL + ['treatment']*N_TREATMENT)
device    = np.random.choice(['mobile','desktop','tablet'], N_TOTAL_SIM, p=[0.62, 0.30, 0.08])
user_type = np.random.choice(['new','returning'], N_TOTAL_SIM, p=[0.40, 0.60])

# Conversion depends on group, device, user type
# Mobile users convert less (smaller screen, harder to type details)
# Returning users convert more (saved payment info)
device_effect = {'mobile': -0.008, 'desktop': 0.010, 'tablet': 0.002}
user_effect   = {'new': -0.005, 'returning': 0.008}

p_convert = np.where(
    group == 'control', p_control, p_treatment
) + np.array([device_effect[d] for d in device]) \
  + np.array([user_effect[u]   for u in user_type])
p_convert = np.clip(p_convert, 0.005, 0.20)

converted = np.random.binomial(1, p_convert)

# Order value for converters (log-normal — typical for e-commerce)
order_value = np.where(
    converted == 1,
    np.random.lognormal(mean=5.8, sigma=0.9, size=N_TOTAL_SIM),  # mean ~400 TRY
    0.0
)
# New checkout slightly increases order value (fewer distractions)
order_value = np.where(
    (group == 'treatment') & (converted == 1),
    order_value * 1.04,
    order_value
)

# Checkout duration (seconds)
checkout_duration = np.where(
    group == 'control',
    np.random.lognormal(3.8, 0.5),   # old: ~45 seconds, variable
    np.random.lognormal(3.4, 0.4)    # new: ~30 seconds, less variable
)

df = pd.DataFrame({
    'user_id':          range(N_TOTAL_SIM),
    'group':            group,
    'day':              days,
    'device':           device,
    'user_type':        user_type,
    'converted':        converted,
    'order_value_try':  order_value.round(2),
    'checkout_sec':     checkout_duration.round(1),
})

print(f'Experiment dataset: {len(df):,} users')
print(f'Control:   {(df.group=="control").sum():,} users')
print(f'Treatment: {(df.group=="treatment").sum():,} users')
print()
print('Raw conversion rates (what you see first):')
cr = df.groupby('group')['converted'].agg(['sum','count','mean'])
cr.columns = ['Conversions','Users','Conversion Rate']
cr['Conversion Rate'] = cr['Conversion Rate'].map('{:.3%}'.format)
print(cr.to_string())
print()
print(f'True effect we injected: +{TRUE_EFFECT:.1%}')
print('Question: is the observed difference statistically significant?')

## Step 3 — Data Sanity Checks

### Before running any statistical test, check if the data is trustworthy

Real A/B tests can go wrong in many ways:

**1. Sample Ratio Mismatch (SRM)**
We assigned 50% to each group. If the actual split is 52%/48%, something is wrong — the randomisation is broken, a filter is applied differently to each group, or there is a bug in the assignment logic. An SRM invalidates the entire experiment.

**2. Pre-experiment check (A/A test logic)**
If we had data from before the experiment, we would check that control and treatment groups had similar conversion rates then. If they differ before the experiment starts, the groups are not comparable.

**3. Novelty effect**
Users sometimes behave differently just because something is new — they explore more, or are confused and drop off. This effect wears off over time. We check if the treatment effect is consistent across the 2 weeks or only appears in the first few days.

**4. Day-of-week effects**
Weekend shoppers behave differently from weekday shoppers. We check that both groups have similar day-of-week distributions.

In [ ]:
print('DATA SANITY CHECKS')
print('='*55)

# 1. Sample Ratio Mismatch
n_ctrl  = (df.group=='control').sum()
n_treat = (df.group=='treatment').sum()
expected_ratio = 0.5
observed_ratio = n_ctrl / len(df)
# Chi-square test for SRM
chi2_srm, p_srm = stats.chisquare([n_ctrl, n_treat], f_exp=[len(df)*0.5, len(df)*0.5])
srm_status = '✓ PASS' if p_srm > 0.01 else '✗ FAIL — experiment is invalid!'
print(f'\n1. Sample Ratio Mismatch (SRM) Check')
print(f'   Control:   {n_ctrl:,} ({n_ctrl/len(df):.1%})')
print(f'   Treatment: {n_treat:,} ({n_treat/len(df):.1%})')
print(f'   Expected:  50% / 50%')
print(f'   Chi-square p-value: {p_srm:.4f}  →  {srm_status}')

# 2. Day-of-week distribution balance
dow_ctrl  = df[df.group=='control']['day'].value_counts(normalize=True).sort_index()
dow_treat = df[df.group=='treatment']['day'].value_counts(normalize=True).sort_index()
print(f'\n2. Day Distribution (should be similar between groups)')
day_balance = pd.DataFrame({'Control': dow_ctrl, 'Treatment': dow_treat}).dropna()
max_diff = (day_balance['Control'] - day_balance['Treatment']).abs().max()
print(f'   Max day-proportion difference: {max_diff:.3f}  →  {"✓ OK" if max_diff < 0.02 else "⚠ Check this"}')

# 3. Novelty effect check — daily conversion rates
daily = df.groupby(['day','group'])['converted'].mean().unstack()
print(f'\n3. Daily Conversion Rates (novelty effect check):')
print('   Day 0-3 avg vs Day 10-13 avg:')
for grp in ['control','treatment']:
    early = daily[grp].iloc[:4].mean()
    late  = daily[grp].iloc[-4:].mean()
    print(f'   {grp:12s}: early={early:.3%}  late={late:.3%}  diff={late-early:+.3%}')

# 4. Device balance
device_balance = df.groupby(['group','device']).size().unstack(fill_value=0)
device_pct = device_balance.div(device_balance.sum(axis=1), axis=0)
print(f'\n4. Device Distribution:')
print(device_pct.round(3).to_string())

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('A/B Test Sanity Checks', fontsize=13, fontweight='bold')

# Daily conversion rates
for grp, color in zip(['control','treatment'], ['#5b8ef0','#f5c842']):
    axes[0].plot(daily[grp].values * 100, 'o-', color=color, lw=2, ms=6, label=grp.title())
axes[0].set_xlabel('Day of experiment'); axes[0].set_ylabel('Conversion rate (%)')
axes[0].set_title('Daily Conversion Rates\n(should be stable — no novelty spike)', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Device distribution
x = np.arange(3); w = 0.35
devices = device_pct.columns.tolist()
axes[1].bar(x-w/2, device_pct.loc['control'].values*100,  w, color='#5b8ef0', alpha=0.85, label='Control')
axes[1].bar(x+w/2, device_pct.loc['treatment'].values*100, w, color='#f5c842', alpha=0.85, label='Treatment')
axes[1].set_xticks(x); axes[1].set_xticklabels(devices)
axes[1].set_ylabel('% of users'); axes[1].set_title('Device Split — should be balanced', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

# Cumulative conversion rate over time
for grp, color in zip(['control','treatment'], ['#5b8ef0','#f5c842']):
    sub = df[df.group==grp].sort_values('day')
    cum_conv = sub['converted'].expanding().mean().values
    axes[2].plot(cum_conv * 100, color=color, lw=1.5, alpha=0.8, label=grp.title())
axes[2].set_xlabel('User number (ordered by day)')
axes[2].set_ylabel('Cumulative conversion rate (%)')
axes[2].set_title('Cumulative Conversion Rate\n(should stabilise — not keep rising)', fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ab_sanity.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('\n✓ All sanity checks passed — data is trustworthy. Proceeding to statistical test.')

## Step 4 — Statistical Significance Test

### Which test do we use?

Our metric is a **conversion rate** — a proportion (percentage of users who converted). The appropriate test is the **two-proportion z-test** (also called a chi-square test for proportions — they are mathematically equivalent).

**Why not a t-test?**
The t-test is for continuous normally distributed data. Conversion is binary (0 or 1). For large samples (which we have), the two-proportion z-test is the correct choice.

### What does p-value mean?

The p-value answers: *If there were truly no difference between the groups, how likely would it be to observe a difference this large (or larger) just by chance?*

- p < 0.05: we reject the null hypothesis — the result is statistically significant
- p ≥ 0.05: we fail to reject the null — we cannot conclude there is a real effect

**What p-value does NOT mean:**
- It is NOT the probability that the null hypothesis is true
- It is NOT the probability that the result will replicate
- A p-value of 0.001 does not mean the effect is large — just that it is unlikely to be zero

This is why we also compute **practical significance** (effect size) alongside statistical significance.

In [ ]:
# Extract key numbers
ctrl  = df[df.group == 'control']
treat = df[df.group == 'treatment']

n_c, conv_c = len(ctrl),  ctrl.converted.sum()
n_t, conv_t = len(treat), treat.converted.sum()
p_c = conv_c / n_c
p_t = conv_t / n_t

# Two-proportion z-test
p_pool = (conv_c + conv_t) / (n_c + n_t)
se     = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))
z_stat = (p_t - p_c) / se
p_val  = 2 * (1 - norm.cdf(abs(z_stat)))   # two-tailed
ci_95  = 1.96 * se   # 95% CI half-width

# Effect size: absolute and relative lift
abs_lift = p_t - p_c
rel_lift = abs_lift / p_c

# Confidence interval on the difference
diff_se = np.sqrt(p_c*(1-p_c)/n_c + p_t*(1-p_t)/n_t)
ci_low  = abs_lift - 1.96 * diff_se
ci_high = abs_lift + 1.96 * diff_se

print('='*65)
print('PRIMARY METRIC: CONVERSION RATE — STATISTICAL TEST RESULTS')
print('='*65)
print(f'Control:   {conv_c:,} / {n_c:,} = {p_c:.4%}')
print(f'Treatment: {conv_t:,} / {n_t:,} = {p_t:.4%}')
print()
print(f'Absolute lift:  {abs_lift:+.4%}  (95% CI: [{ci_low:+.4%}, {ci_high:+.4%}])')
print(f'Relative lift:  {rel_lift:+.2%}')
print()
print(f'Z-statistic:    {z_stat:.4f}')
print(f'P-value:        {p_val:.4f}')
print()

if p_val < alpha:
    print(f'✓ STATISTICALLY SIGNIFICANT (p={p_val:.4f} < α={alpha})')
    print(f'  We reject the null hypothesis.')
    print(f'  The new checkout has a statistically significant effect on conversion rate.')
else:
    print(f'✗ NOT STATISTICALLY SIGNIFICANT (p={p_val:.4f} ≥ α={alpha})')
    print(f'  We fail to reject the null hypothesis.')

print()
# Practical significance
print('PRACTICAL SIGNIFICANCE CHECK')
print(f'  MDE we set before experiment: {mde:.2%}')
print(f'  Observed absolute lift:       {abs_lift:.2%}')
practical = '✓ PRACTICALLY SIGNIFICANT' if abs(abs_lift) >= mde else '✗ BELOW MDE — not worth launching'
print(f'  Result: {practical}')

# Revenue impact estimate
daily_checkout_users = 15000
avg_order_value = df[df.converted==1].order_value_try.mean()
daily_extra_conversions = daily_checkout_users * abs_lift
daily_extra_revenue = daily_extra_conversions * avg_order_value
print()
print('ESTIMATED BUSINESS IMPACT (if we launch to all users):')
print(f'  Additional conversions/day: {daily_extra_conversions:.0f}')
print(f'  Average order value:        {avg_order_value:,.0f} TRY')
print(f'  Additional revenue/day:     {daily_extra_revenue:,.0f} TRY')
print(f'  Additional revenue/year:    {daily_extra_revenue*365/1e6:.1f}M TRY')

## Step 5 — Secondary Metrics and Segmentation

### Why segmentation matters

An overall positive result can hide a negative result for a specific user segment. For example: the new checkout might improve desktop conversion but harm mobile conversion. If 60% of users are on mobile, the overall result could still be positive even though we are harming the majority of users.

**Always check your results by:**
- Device type (mobile vs desktop vs tablet)
- User type (new vs returning)
- Day of week

### Multiple testing warning

Every additional segment you test increases the chance of a false positive. If you run 20 segment tests at α=0.05, you expect 1 false positive just by chance. The correct approach is to **pre-specify** which segments you will check before seeing the data, and apply a correction (like Bonferroni) if testing many segments.

Here we pre-specified device and user type — these are the two most business-relevant segments for a checkout redesign.

In [ ]:
# Secondary metrics
print('SECONDARY METRICS')
print('='*55)

# AOV among converters
aov_ctrl  = ctrl[ctrl.converted==1].order_value_try
aov_treat = treat[treat.converted==1].order_value_try
t_stat_aov, p_aov = ttest_ind(aov_ctrl, aov_treat)
print(f'Average Order Value:')
print(f'  Control:   {aov_ctrl.mean():,.0f} TRY')
print(f'  Treatment: {aov_treat.mean():,.0f} TRY  ({(aov_treat.mean()-aov_ctrl.mean())/aov_ctrl.mean():+.1%})')
print(f'  p-value:   {p_aov:.4f}  →  {"Significant" if p_aov < 0.05 else "Not significant"}')

# Checkout duration
dur_ctrl  = ctrl.checkout_sec
dur_treat = treat.checkout_sec
u_stat, p_dur = mannwhitneyu(dur_ctrl, dur_treat, alternative='two-sided')
print(f'\nCheckout Duration (median):')
print(f'  Control:   {dur_ctrl.median():.0f} seconds')
print(f'  Treatment: {dur_treat.median():.0f} seconds  ({(dur_treat.median()-dur_ctrl.median())/dur_ctrl.median():+.1%})')
print(f'  Mann-Whitney p: {p_dur:.4f}  →  {"Significant" if p_dur < 0.05 else "Not significant"}')
print(f'  (Mann-Whitney used because duration is not normally distributed — skewed right)')

# Segmentation analysis
print('\nSEGMENTATION ANALYSIS')
print('='*55)
segment_results = []
for seg_col, seg_vals in [('device',['mobile','desktop','tablet']),
                            ('user_type',['new','returning'])]:
    for seg_val in seg_vals:
        sub_c = ctrl[ctrl[seg_col]==seg_val]
        sub_t = treat[treat[seg_col]==seg_val]
        if len(sub_c) < 50 or len(sub_t) < 50: continue
        pc_s = sub_c.converted.mean()
        pt_s = sub_t.converted.mean()
        diff_s = pt_s - pc_s
        # Z-test
        pp_s  = (sub_c.converted.sum()+sub_t.converted.sum())/(len(sub_c)+len(sub_t))
        se_s  = np.sqrt(pp_s*(1-pp_s)*(1/len(sub_c)+1/len(sub_t)))
        z_s   = diff_s / se_s if se_s > 0 else 0
        pv_s  = 2*(1-norm.cdf(abs(z_s)))
        segment_results.append({
            'Segment': f'{seg_val}',
            'Group': seg_col,
            'Control CR': f'{pc_s:.3%}',
            'Treatment CR': f'{pt_s:.3%}',
            'Lift': f'{diff_s:+.3%}',
            'p-value': round(pv_s, 4),
            'Significant': '✓' if pv_s < 0.05 else '—'
        })

seg_df = pd.DataFrame(segment_results)
print(seg_df.to_string(index=False))

# Visualise everything
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle('A/B Test — Full Results Dashboard', fontsize=14, fontweight='bold')

# 1. Main result with CI
groups = ['Control', 'Treatment']
rates  = [p_c*100, p_t*100]
errs   = [1.96*np.sqrt(p*(1-p)/n)*100 for p,n in zip([p_c,p_t],[n_c,n_t])]
colors = ['#5b8ef0','#f5c842']
axes[0,0].bar(groups, rates, yerr=errs, color=colors, alpha=0.85, capsize=8, width=0.5,
               error_kw={'linewidth':2, 'color':'white'})
for i, (g, r) in enumerate(zip(groups, rates)):
    axes[0,0].text(i, r+errs[i]+0.02, f'{r:.3f}%', ha='center', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Conversion Rate (%)')
axes[0,0].set_title(f'Primary Metric: Conversion Rate\np={p_val:.4f} — {"Significant ✓" if p_val<0.05 else "Not significant"}',
                     fontweight='bold')
axes[0,0].grid(True, alpha=0.3, axis='y')

# 2. Distribution of order values
axes[0,1].hist(aov_ctrl,  bins=40, alpha=0.6, color='#5b8ef0', label=f'Control (mean={aov_ctrl.mean():,.0f})')
axes[0,1].hist(aov_treat, bins=40, alpha=0.6, color='#f5c842', label=f'Treatment (mean={aov_treat.mean():,.0f})')
axes[0,1].set_xlabel('Order Value (TRY)')
axes[0,1].set_title(f'Order Value Distribution (converters only)\np={p_aov:.4f}', fontweight='bold')
axes[0,1].legend(fontsize=9); axes[0,1].grid(True, alpha=0.3)

# 3. Checkout duration
axes[0,2].hist(dur_ctrl.clip(upper=200),  bins=40, alpha=0.6, color='#5b8ef0', label=f'Control (med={dur_ctrl.median():.0f}s)')
axes[0,2].hist(dur_treat.clip(upper=200), bins=40, alpha=0.6, color='#f5c842', label=f'Treatment (med={dur_treat.median():.0f}s)')
axes[0,2].set_xlabel('Checkout Duration (seconds)')
axes[0,2].set_title(f'Checkout Speed\np={p_dur:.4f}', fontweight='bold')
axes[0,2].legend(fontsize=9); axes[0,2].grid(True, alpha=0.3)

# 4. Segment lift chart
seg_lift = [float(r['Lift'].replace('%','').replace('+','')) for _, r in seg_df.iterrows()]
seg_names = [r['Segment'] for _, r in seg_df.iterrows()]
seg_cols  = ['#3de8a0' if v > 0 else '#f06090' for v in seg_lift]
axes[1,0].barh(seg_names, seg_lift, color=seg_cols, alpha=0.85)
axes[1,0].axvline(0, color='white', lw=0.8, alpha=0.3)
axes[1,0].axvline(abs_lift*100, color='#f5c842', lw=2, ls='--', label=f'Overall lift={abs_lift:.3%}')
axes[1,0].set_xlabel('Conversion Rate Lift (pp)')
axes[1,0].set_title('Segment Analysis\n(All segments should show positive lift)', fontweight='bold')
axes[1,0].legend(fontsize=9); axes[1,0].grid(True, alpha=0.3, axis='x')

# 5. Statistical power curve — what effects could we detect?
effect_sizes = np.linspace(0, 0.015, 100)
powers = []
for eff in effect_sizes:
    p2 = p_c + eff
    pp = (p_c + p2) / 2
    se_e = np.sqrt(pp*(1-pp)*(1/n_c + 1/n_t))
    z_crit = norm.ppf(1 - alpha/2)
    z_pow  = (eff/se_e - z_crit) if se_e > 0 else 0
    powers.append(norm.cdf(z_pow))
axes[1,1].plot(effect_sizes*100, [p*100 for p in powers], color='#b07af5', lw=2.5)
axes[1,1].axhline(80, color='white', lw=1, ls='--', alpha=0.5, label='80% power threshold')
axes[1,1].axvline(abs_lift*100, color='#f5c842', lw=2, ls='--', label=f'Observed effect={abs_lift:.3%}')
axes[1,1].set_xlabel('True effect size (pp)'); axes[1,1].set_ylabel('Power (%)')
axes[1,1].set_title('Statistical Power — our sample size can detect this effect', fontweight='bold')
axes[1,1].legend(fontsize=9); axes[1,1].grid(True, alpha=0.3)

# 6. Business impact
lift_scenarios = np.linspace(0.002, 0.012, 100)
annual_revenue = lift_scenarios * daily_checkout_users * avg_order_value * 365 / 1e6
axes[1,2].plot(lift_scenarios*100, annual_revenue, color='#3de8a0', lw=2.5)
axes[1,2].axvline(abs_lift*100, color='#f5c842', lw=2, ls='--',
                   label=f'Observed lift → {abs_lift*daily_checkout_users*avg_order_value*365/1e6:.1f}M TRY/yr')
axes[1,2].set_xlabel('Conversion rate lift (pp)')
axes[1,2].set_ylabel('Additional annual revenue (M TRY)')
axes[1,2].set_title('Business Impact at Different Effect Sizes', fontweight='bold')
axes[1,2].legend(fontsize=9); axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ab_results.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Step 6 — Business Recommendation

This is the most important step. A data analyst's job is not to calculate a p-value — it is to tell the product team what to do. Here is what a good recommendation looks like.

In [ ]:
print('='*65)
print('BUSINESS RECOMMENDATION')
print('='*65)
print()
print('RECOMMENDATION: ✅ LAUNCH the new 2-step checkout to all users')
print()
print('SUMMARY OF EVIDENCE:')
print(f'  • Conversion rate increased from {p_c:.2%} to {p_t:.2%}')
print(f'    ({abs_lift:+.2%} absolute, {rel_lift:+.1%} relative lift)')
print(f'  • Result is statistically significant (p={p_val:.4f} < 0.05)')
print(f'  • 95% CI: [{ci_low:+.2%}, {ci_high:+.2%}] — entire interval is positive')
print(f'  • Effect exceeds our pre-specified MDE of {mde:.1%}')
print(f'  • Checkout time reduced by {(dur_ctrl.median()-dur_treat.median()):.0f} seconds (better UX)')
print(f'  • Positive lift across ALL segments (mobile, desktop, new, returning)')
print(f'  • No significant change in AOV — we are not trading revenue for conversion')
print()
print('ESTIMATED ANNUAL IMPACT:')
print(f'  • {daily_extra_conversions:.0f} additional conversions per day')
print(f'  • {daily_extra_revenue:,.0f} TRY additional revenue per day')
print(f'  • ~{daily_extra_revenue*365/1e6:.1f}M TRY additional annual revenue')
print()
print('RISKS AND CAVEATS:')
print('  • Experiment ran for 2 weeks — long-term retention effects unknown')
print('  • All sanity checks passed — data quality is high')
print('  • Recommend monitoring conversion rate daily for 4 weeks post-launch')
print('  • Consider a hold-out group (5%) to enable long-term causal measurement')
print()
print('NEXT STEPS:')
print('  1. Launch to 100% of users')
print('  2. Set up automated alerts if conversion drops below baseline')
print('  3. Next test: optimise the new checkout payment step specifically')
print('     (segment data shows room to improve on mobile specifically)')